# Dataloader visualization: original files vs. `PairDataset` output

For a chosen **loader sample index** in each dataset this notebook shows, side by side:

- **top row, original dataset**: the frames and the event stream read straight from the downloaded files
  (HQ-EVFI / BS-ERGB `.png` + `.npz`, CED `.bag`, EDS `images/` + `events.h5`), cut to the same frames, time window and crop;
- **bottom row, dataloader**: what `data.PairDataset` returns for that index (`i0`, `i1`, the event list
  `ev_pix / ev_tau / ev_pol`, and their timing).

A second figure shows the rest of the sample: the student's context frames, the hidden frames, and the teacher's voxel grid.

Below each sample a **check list** compares the two numerically:

| Check | Expected |
|---|---|
| loader vs converted data on disk: frames and events | identical (exact) |
| loader vs original files: events in the window and crop | same count and same per-pixel ON/OFF counts |
| loader vs original files: frame pixels | identical (EDS: after warping the stored RGB into the event camera with the calibration) |
| voxel grid | holds every event exactly once |

Event colours: **red = ON**, **blue = OFF**, white = both at the same pixel. Run from the repository root
(where this notebook lives), on the branch `multi-dataset-loader`, after converting the data (README section 4).

In [ ]:
# ---- settings: edit these --------------------------------------------------------------------------------
CONFIG = "config.yaml"
SPLIT = "train"              # train | test  (BS-ERGB also has val)
TRAIN_MODE = False           # False: centre crop, no flip (as in validation); True: random crop + flip as in training
SEED = 0                     # with TRAIN_MODE, the crop / flip of sample n is drawn with seed SEED + n
CROP = None                  # None: data.crop of the config

DATA = {                     # conv: converted data the loader reads (holds train/ test/); raw: the original download
    "hqevfi": dict(conv="data/hqevfi", raw="HQ-EVFI/raw"),   # HQ-EVFI, BS-ERGB: the originals are found through the
    "bsergb": dict(conv="data/bsergb", raw="BS-ERGB"),        # symlinks the converter made, so raw is not used
    "ced":    dict(conv="data/ced",    raw="CED"),           # folder holding the .bag files (searched recursively)
    "eds":    dict(conv="data/eds",    raw="EDS/EDS", calib=None),  # raw: the extracted sequence folders; calib: the
                                                                        # --calib yaml if you converted with one
}
INDEX = {                    # loader sample indices to show per dataset (taken modulo the dataset length)
    "hqevfi": [0, 1500],
    "bsergb": [0, 800],
    "ced":    [0, 400],
    "eds":    [0, 2000],
}

In [ ]:
# ---- helpers: original-file readers, plots, checks ----------------------------------------------------------
import contextlib, glob, importlib.util, io, os, re, sys, tempfile
import numpy as np
import torch
import yaml
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

sys.path.insert(0, os.getcwd())
from data import PairDataset, Sequence, data_sets

natural = lambda f: [int(s) if s.isdigit() else s for s in re.split(r"(\d+)", os.path.basename(f))]
SCALES = (1.0, 1e-3, 1e-6, 1e-9)
IMG = (".png", ".jpg", ".jpeg")


def load_script(path, argv):
    # import a converter script for its reader functions (it runs on nothing: empty --raw, prints hidden)
    old = sys.argv
    sys.argv = [path] + argv
    try:
        spec = importlib.util.spec_from_file_location(os.path.basename(path)[:-3] + "_mod", path)
        mod = importlib.util.module_from_spec(spec)
        with contextlib.redirect_stdout(io.StringIO()):
            spec.loader.exec_module(mod)
        return mod
    finally:
        sys.argv = old


EMPTY = tempfile.mkdtemp()
_mods = {}
def ced_mod():
    _mods.setdefault("ced", load_script("CED/ced.py", ["--raw", EMPTY]))
    return _mods["ced"]
def eds_mod():
    _mods.setdefault("eds", load_script("EDS/eds.py", ["--raw", EMPTY, "--archive", "--dry"]))
    return _mods["eds"]


def rgb_u8(f):
    return np.asarray(Image.open(f).convert("RGB"))


def traced(seq, k):
    # converted frames are symlinks to the original PNGs (unless the converter ran with --copy)
    f = sorted(glob.glob(os.path.join(seq, "frames", "*.png")))[k]
    if not os.path.islink(f):
        raise RuntimeError("converted frames are copies (--copy), so the original images cannot be traced; "
                           "convert without --copy to use this notebook on this dataset")
    r = os.path.realpath(f)
    if not os.path.exists(r):
        raise RuntimeError(f"original image moved or deleted: {r}")
    return r


def chunk_seconds(ts, fps):
    # raw event times of consecutive per-frame files -> seconds from the first event, whatever the unit;
    # files whose times restart at 0 are laid end to end at 1/fps (as the converter does)
    lo = np.array([t.min() if len(t) else np.nan for t in ts], dtype=np.float64)
    hi = np.array([t.max() if len(t) else np.nan for t in ts], dtype=np.float64)
    span = np.nanmedian(hi - lo)
    s = next((s for s in SCALES if span * s <= 1.5 / fps), SCALES[-1])
    restart = np.any(lo[1:] < hi[:-1] - 0.5 * span)
    out = [(k / fps + t.astype(np.float64) * s) if restart else t.astype(np.float64) * s for k, t in enumerate(ts)]
    t = np.concatenate(out) if out else np.zeros(0)
    return t - (t.min() if len(t) else 0), ("per-file times laid end to end" if restart else f"unit {s:g} s")


# every reader returns: img0, img1 (original frames, full size), rgb_box (y0, x0, h, w) of the loader crop in the
# original image or None, ev (x, y, t seconds from the first event / from I0, p), ev_box of the crop in sensor
# pixels, span (I0 -> I1 in seconds, None if the files do not say), files (what was read), notes
def raw_hqevfi(p, info):
    seq, i, j = info["seq"], info["i"], info["j"]
    ri, rj = traced(seq, i), traced(seq, j)
    rawseq = os.path.dirname(os.path.dirname(ri))
    name = os.path.basename(rawseq)
    lists = os.path.join(p["conv"], "dataset_dict.py")
    spec = importlib.util.spec_from_file_location("dataset_dict", lists)
    meta = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(meta)
    shifted = int(name in meta.EVSneg3 and os.path.isdir(os.path.join(rawseq, "RGB-EVS_EVSneg3ms")))
    imgs = sorted((f for f in glob.glob(os.path.join(rawseq, "visual_RGB", "*")) if f.lower().endswith(IMG)), key=natural)
    q = [os.path.realpath(f) for f in imgs].index(ri) - shifted  # event file k sits between image k and k+1
    efiles = sorted(glob.glob(os.path.join(rawseq, "RGB-EVS_EVSneg3ms" if shifted else "RGB-EVS", "*.npz")), key=natural)
    use = efiles[q:q + (j - i)]
    ch = [np.load(f) for f in use]
    t, how = chunk_seconds([np.asarray(c["t"]).ravel() for c in ch], 142.0)
    ev = dict(x=np.concatenate([np.asarray(c["x"]).ravel() for c in ch]).round().astype(np.int64),
              y=np.concatenate([np.asarray(c["y"]).ravel() for c in ch]).round().astype(np.int64), t=t,
              p=np.concatenate([np.asarray(c["p"]).ravel() for c in ch]))
    box = (info["y0"], info["x0"], info["h"], info["w"])
    return dict(img0=rgb_u8(ri), img1=rgb_u8(rj), rgb_box=box, ev=ev, ev_box=box, span=None,
                files=[ri, rj] + [use[0], "...", use[-1]],
                notes=[f"{len(use)} event files ({os.path.basename(os.path.dirname(use[0]))}), times: {how}; "
                       "time axis starts at the first event of the gap"])


def raw_bsergb(p, info):
    seq, i, j = info["seq"], info["i"], info["j"]
    ri, rj = traced(seq, i), traced(seq, j)
    rawseq = os.path.dirname(os.path.dirname(ri))
    imgs = sorted(glob.glob(os.path.join(rawseq, "images", "*.png")))
    r = [os.path.realpath(f) for f in imgs].index(ri)  # events/k.npz holds the events between image k and k+1
    efiles = sorted(glob.glob(os.path.join(rawseq, "events", "*.npz")))
    use = efiles[r:r + (j - i)]
    ch = [np.load(f) for f in use]
    img0 = rgb_u8(ri)
    x = np.concatenate([np.asarray(c["x"]).ravel() for c in ch]).astype(np.float64)
    y = np.concatenate([np.asarray(c["y"]).ravel() for c in ch]).astype(np.float64)
    sub = 32.0 if x.max() > 1.5 * img0.shape[1] else 1.0
    t, how = chunk_seconds([np.asarray(c["timestamp"]).ravel() for c in ch], Sequence(seq).fps)
    ev = dict(x=np.round(x / sub).astype(np.int64), y=np.round(y / sub).astype(np.int64), t=t,
              p=np.concatenate([np.asarray(c["polarity"]).ravel() for c in ch]))
    box = (info["y0"], info["x0"], info["h"], info["w"])
    return dict(img0=img0, img1=rgb_u8(rj), rgb_box=box, ev=ev, ev_box=box, span=None,
                files=[ri, rj] + [use[0], "...", use[-1]],
                notes=[f"{len(use)} event files, x/y unit 1/{sub:.0f} px, times: {how}; "
                       "time axis starts at the first event of the gap"])


_bags = {}
def raw_ced(p, info):
    seq, i, j = info["seq"], info["i"], info["j"]
    name = os.path.basename(seq)
    bags = sorted(glob.glob(os.path.join(p["raw"], "**", name + ".bag"), recursive=True))
    if not bags:
        raise RuntimeError(f"{name}.bag not found under {p['raw']}")
    if bags[0] not in _bags:  # whole bag, once per notebook session
        with contextlib.redirect_stdout(io.StringIO()):
            _bags[bags[0]] = ced_mod().read(bags[0])
    ev, frames, pattern = _bags[bags[0]]
    base = int(ev["s"].min())
    t = (ev["s"] - base).astype(np.float64) + ev["ns"] * 1e-9
    ts = np.array([sec - base + nsec * 1e-9 for sec, nsec, _, _ in frames])
    kept = np.nonzero((ts >= t.min()) & (ts <= t.max()))[0]  # frames inside the event stream, as converted
    conv_ts = np.load(os.path.join(seq, "frame_ts.npy"))
    drift = np.abs((ts[kept] - ts[kept[0]]) - (conv_ts - conv_ts[0])).max() if len(kept) == len(conv_ts) else np.inf
    a, b = kept[i], kept[j]
    m = (t >= ts[a]) & (t < ts[b])
    img = lambda k: ced_mod().rgb(frames[k][2], frames[k][3], pattern)
    box = (info["y0"], info["x0"], info["h"], info["w"])
    return dict(img0=img(a), img1=img(b), rgb_box=box,
                ev=dict(x=ev["x"][m].astype(np.int64), y=ev["y"][m].astype(np.int64), t=t[m] - ts[a], p=ev["p"][m]),
                ev_box=box, span=ts[b] - ts[a], files=[bags[0] + f"  (frames {a}, {b} of {len(frames)})"],
                notes=[f"frames from {'image_raw (' + pattern + ', demosaiced)' if pattern else 'image_color'}; "
                       f"frame times bag vs converted: max difference {drift * 1e6:.2f} us"])


def h5_events(path):
    import h5py
    try:
        import hdf5plugin  # noqa: F401
    except ImportError:
        pass
    f = h5py.File(path, "r")
    for keys in (("events/x", "events/y", "events/t", "events/p"), ("x", "y", "t", "p"), ("xs", "ys", "ts", "ps"),
                 ("events/xs", "events/ys", "events/ts", "events/ps")):
        if all(k in f for k in keys):
            off = float(f["t_offset"][()]) if "t_offset" in f else 0.0
            return f, [f[k] for k in keys], off
    raise KeyError(f"{path}: no x/y/t/p datasets, has {list(f.keys())}")


def bisect(dset, v):
    # first index with dset[index] >= v in a sorted on-disk array (a few reads, no full load)
    lo, hi = 0, len(dset)
    while lo < hi:
        mid = (lo + hi) // 2
        if dset[mid] < v:
            lo = mid + 1
        else:
            hi = mid
    return lo


def raw_eds(p, info):
    E = eds_mod()
    seq, i, j = info["seq"], info["i"], info["j"]
    name = os.path.basename(seq)
    cands = [d for d in glob.glob(os.path.join(p["raw"], "**", name), recursive=True)
             if glob.glob(os.path.join(d, "**", "events*"), recursive=True)]
    if not cands:
        raise RuntimeError(f"sequence folder {name} with an events file not found under {p['raw']}")
    rawseq = cands[0]
    al = yaml.safe_load(open(os.path.join(seq, "align.yaml")))
    Y0, X0 = al["crop_yxyx"][:2]
    imgs = E.find_images(rawseq)
    ts, tsrc = E.read_times(rawseq, imgs)
    imgs = imgs[:len(ts)]
    s = next(s for s in SCALES if 2e-3 <= np.median(np.diff(ts)) * s <= 0.5)
    ts = ts * s
    h5 = sorted(glob.glob(os.path.join(rawseq, "**", "events*.h5"), recursive=True))
    if h5:
        fh, (dx, dy, dt_, dp), off = h5_events(h5[0])
        first, last = float(dt_[0]) + off, float(dt_[-1]) + off
    else:  # events.txt: small enough to load
        x_, y_, t_, p_, _ = E.read_events(rawseq)
        dx, dy, dt_, dp, off, first, last = x_, y_, t_, p_, 0.0, float(t_[0]), float(t_[-1])
    overlap = lambda se: (min(last * se, ts[-1]) - max(first * se, ts[0])) / (ts[-1] - ts[0])
    se = max(SCALES, key=overlap)
    kept = np.nonzero((ts >= first * se) & (ts <= last * se))[0]
    conv_ts = np.load(os.path.join(seq, "frame_ts.npy"))
    drift = np.abs((ts[kept] - ts[kept[0]]) - (conv_ts - conv_ts[0])).max() if len(kept) == len(conv_ts) else np.inf
    a, b = kept[i], kept[j]
    lo_raw, hi_raw = ts[a] / se - off, ts[b] / se - off  # window in the file's own time unit
    if h5:
        ia, ib = bisect(dt_, lo_raw), bisect(dt_, hi_raw)
        ev = dict(x=np.asarray(dx[ia:ib]).astype(np.int64), y=np.asarray(dy[ia:ib]).astype(np.int64),
                  t=(np.asarray(dt_[ia:ib]).astype(np.float64) + off) * se - ts[a], p=np.asarray(dp[ia:ib]))
    else:
        m = (dt_ >= lo_raw) & (dt_ < hi_raw)
        ev = dict(x=dx[m].astype(np.int64), y=dy[m].astype(np.int64), t=(dt_[m] + off) * se - ts[a], p=dp[m])
    # the stored RGB sees the scene through another lens: warp it into the event camera with the dataset calibration
    # (flip from align.yaml, same maps as EDS/eds.py), then cut the converter's crop, so it lines up with the events
    y1, x1 = al["crop_yxyx"][2:]
    mx, my = eds_maps(p, rgb_u8(imgs[a]).shape)
    mx, my = mx[Y0:y1, X0:x1], my[Y0:y1, X0:x1]
    warp = lambda f: E.sample(E.flipped(rgb_u8(f), al["flip"]), mx, my)
    ev["x"], ev["y"] = ev["x"] - X0, ev["y"] - Y0  # sensor pixels -> the same aligned frame
    box = (info["y0"], info["x0"], info["h"], info["w"])
    calib = "calibration " + p["calib"] if p.get("calib") else "dataset calibration"
    return dict(img0=warp(imgs[a]), img1=warp(imgs[b]), rgb_box=box, ev=ev, ev_box=box, span=ts[b] - ts[a],
                files=[imgs[a], imgs[b], h5[0] if h5 else "events.txt"],
                notes=[f"original RGB warped to the event camera (flip '{al['flip']}', {calib}) and cut to the converter's "
                       f"crop y{Y0}:{y1} x{X0}:{x1}; events are untouched sensor pixels, shifted by that crop",
                       f"frame times ({tsrc}) vs converted: max difference {drift * 1e6:.2f} us; "
                       f"converter alignment: residual shift {tuple(al['residual_shift'])}, score {al['score']:.3f}"])


_maps = {}
def eds_maps(p, shape):
    # event pixel -> stored RGB pixel, as EDS/eds.py computes it (cached per calibration and image size)
    E = eds_mod()
    key = (p.get("calib"), shape[:2])
    if key not in _maps:
        cal = yaml.safe_load(open(p["calib"])) if p.get("calib") else E.cal
        eW, eH = cal["cam1"]["resolution"]
        rW, rH = cal["cam0"]["resolution"]
        mx, my = E.rgb_map(cal, eW, eH)
        H, W = shape[:2]
        if (W, H) != (rW, rH):
            mx, my = (mx + 0.5) * W / rW - 0.5, (my + 0.5) * H / rH - 0.5
        _maps[key] = (mx, my)
    return _maps[key]


RAW = dict(hqevfi=raw_hqevfi, bsergb=raw_bsergb, ced=raw_ced, eds=raw_eds)


# ---- drawing ------------------------------------------------------------------------------------------------
def ev_image(x, y, p, h, w, bg=None):
    on = np.bincount(y[p > 0] * w + x[p > 0], minlength=h * w).reshape(h, w)
    off = np.bincount(y[p <= 0] * w + x[p <= 0], minlength=h * w).reshape(h, w)
    base = 0.35 * bg.astype(np.float32).mean(-1) / 255 if bg is not None else np.full((h, w), 0.12, np.float32)
    img = np.repeat(base[..., None], 3, -1)
    img[(on > 0) & (off == 0)] = (1.0, 0.22, 0.18)
    img[(off > 0) & (on == 0)] = (0.25, 0.5, 1.0)
    img[(on > 0) & (off > 0)] = (1.0, 1.0, 1.0)
    return img, on, off


def to_u8(t):
    return (np.asarray(t, dtype=np.float32).transpose(1, 2, 0) * 255).round().clip(0, 255).astype(np.uint8)


def cut(img, box, flip):
    y0, x0, h, w = box
    c = img[y0:y0 + h, x0:x0 + w]
    return c[:, ::-1] if flip else c


def into_crop(ev, box, flip):
    # original sensor coordinates -> the loader's crop (and its flip)
    y0, x0, h, w = box
    x, y = ev["x"] - x0, ev["y"] - y0
    m = (x >= 0) & (x < w) & (y >= 0) & (y < h)
    x = w - 1 - x[m] if flip else x[m]
    return x, y[m], np.where(ev["p"][m] > 0, 1, -1), ev["t"][m]


def time_hist(ax, t, p, span, title, marks=()):
    bins = np.linspace(0, span, 61)
    ax.stairs(np.histogram(t[p > 0], bins)[0], bins * 1e3, color=(0.9, 0.2, 0.15), label=f"ON {int((p > 0).sum())}")
    ax.stairs(np.histogram(t[p <= 0], bins)[0], bins * 1e3, color=(0.2, 0.45, 0.95), label=f"OFF {int((p <= 0).sum())}")
    for m in marks:
        ax.axvline(m * 1e3, color="0.5", lw=0.8, ls=":")
    ax.set_xlabel("time from I0 (ms)")
    ax.set_ylabel("events per bin")
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8, loc="upper right")


def show_img(ax, img, title, box=None, color="yellow"):
    ax.imshow(img, interpolation="nearest")
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    if box is not None:
        y0, x0, h, w = box
        ax.add_patch(Rectangle((x0 - 0.5, y0 - 0.5), w, h, fill=False, ec=color, lw=1.5))


# ---- one sample ---------------------------------------------------------------------------------------------
def loader_sample(name, n):
    ds = DS[name]
    n = n % len(ds)
    torch.manual_seed(SEED + n)
    item = ds[n]
    return n, item, dict(ds.last)


def disk_check(item, info):
    # loader output vs the converted sequence on disk: same frames, same events (exact)
    s = Sequence(info["seq"])
    box = (info["y0"], info["x0"], info["h"], info["w"])
    f0 = cut(to_u8(s.frame(info["i"])), box, info["flip"])
    f1 = cut(to_u8(s.frame(info["j"])), box, info["flip"])
    e = s.events(info["t0"], info["t1"])
    x, y, p, t = into_crop(dict(x=e["x"].astype(np.int64), y=e["y"].astype(np.int64), p=e["p"], t=e["t"]), box, info["flip"])
    w = info["w"]
    tau = ((t - info["t0"]) / (info["t1"] - info["t0"])).astype(np.float32)
    ref = np.lexsort((p, tau, y * w + x))
    pix, tau_l, pol = np.asarray(item["ev_pix"]), np.asarray(item["ev_tau"]), np.asarray(item["ev_pol"])
    got = np.lexsort((pol, tau_l, pix))
    same = len(pix) == len(x) and np.array_equal(pix[got], (y * w + x)[ref]) and np.array_equal(pol[got], p[ref]) \
        and np.allclose(tau_l[got], tau[ref], atol=1e-6)
    return dict(frames=bool(np.array_equal(f0, to_u8(item["i0"])) and np.array_equal(f1, to_u8(item["i1"]))),
                events=same, n_disk=len(x))


def compare(name, n):
    n, item, info = loader_sample(name, n)
    h, w, flip = info["h"], info["w"], info["flip"]
    dt = float(np.asarray(item["dt"]))
    i0, i1 = to_u8(item["i0"]), to_u8(item["i1"])
    pix, tau, pol = np.asarray(item["ev_pix"]), np.asarray(item["ev_tau"]), np.asarray(item["ev_pol"])
    ly, lx = pix // w, pix % w
    mid_tau = np.asarray(item["mid_tau"])
    seqname = os.path.basename(info["seq"])
    try:
        raw, err = RAW[name](DATA[name], info), None
    except Exception as e:  # still show the loader side
        raw, err = None, f"{type(e).__name__}: {e}"

    fig, ax = plt.subplots(2, 5, figsize=(24, 9.5), gridspec_kw=dict(width_ratios=[1.35, 1, 1, 1, 1.25]))
    fig.suptitle(f"{name} | loader index {n} | {seqname} frames {info['i']} -> {info['j']} (skip {info['skip']}) | "
                 f"gap {dt * 1e3:.2f} ms | crop y{info['y0']} x{info['x0']} {w}x{h}{' | flipped' if flip else ''}",
                 fontsize=13, y=0.995)
    lev, lon, loff = ev_image(lx, ly, pol, h, w, i0)
    checks = []
    if raw is not None:
        rx, ry, rp, rt = into_crop(raw["ev"], raw["ev_box"], flip)
        span = raw["span"] if raw["span"] is not None else max(dt, rt.max() if len(rt) else dt)
        if raw["rgb_box"] is not None:
            show_img(ax[0, 0], raw["img0"], f"original I0 ({raw['img0'].shape[1]}x{raw['img0'].shape[0]}), crop box", raw["rgb_box"])
            c0, c1 = cut(raw["img0"], raw["rgb_box"], flip), cut(raw["img1"], raw["rgb_box"], flip)
            show_img(ax[0, 1], c0, "original I0, crop" + (" (mirrored)" if flip else ""))
            show_img(ax[0, 2], c1, "original I1, crop" + (" (mirrored)" if flip else ""))
            bg = c0
        else:
            full, _, _ = ev_image(raw["ev"]["x"], raw["ev"]["y"], raw["ev"]["p"], *raw["img0"].shape[:2])
            show_img(ax[0, 0], full, "original events on the full sensor, crop box", raw["ev_box"])
            show_img(ax[0, 1], raw["img0"], "original I0 as stored (not aligned)")
            show_img(ax[0, 2], raw["img1"], "original I1 as stored (not aligned)")
            bg = None
        rev, ron, roff = ev_image(rx, ry, rp, h, w, bg)
        show_img(ax[0, 3], rev, f"original events in the crop: {len(rx)}")
        time_hist(ax[0, 4], rt, rp, span, "original event times")
        dk = disk_check(item, info)
        same_px = np.array_equal(ron, lon) and np.array_equal(roff, loff)
        checks += [("loader frames == converted frames on disk", dk["frames"], ""),
                   ("loader events == converted events on disk", dk["events"], f"{len(pix)} vs {dk['n_disk']}"),
                   ("loader events == original events (count)", len(rx) == len(pix), f"{len(pix)} vs {len(rx)}"),
                   ("loader ON/OFF counts per pixel == original", same_px,
                    f"max |diff| {int(max(np.abs(ron - lon).max(), np.abs(roff - loff).max()))}")]
        if raw["rgb_box"] is not None:
            d = max(np.abs(c0.astype(int) - i0.astype(int)).max(), np.abs(c1.astype(int) - i1.astype(int)).max())
            checks.append(("loader I0/I1 pixels == original crop", d == 0, f"max |diff| {d}"))
        notes = raw["notes"]
    else:
        for k in range(5):
            ax[0, k].axis("off")
        ax[0, 2].text(0.5, 0.5, "original files not read:\n" + err, ha="center", va="center", wrap=True, fontsize=10)
        span, notes = dt, []
    vox = np.asarray(item["voxel"])
    checks.append(("voxel grid holds every event once", int(vox.sum()) == len(pix), f"{int(vox.sum())} vs {len(pix)}"))
    checks.append(("event times inside the gap", bool(len(tau) == 0 or (tau.min() >= 0 and tau.max() <= 1)), ""))
    if len(pix) == 0:  # nothing to compare: the event checks pass trivially
        checks.append(("no events in this crop: event checks say nothing, pick another index", None, ""))

    show_img(ax[1, 1], i0, f"loader i0  {tuple(np.asarray(item['i0']).shape)}")
    show_img(ax[1, 2], i1, "loader i1")
    show_img(ax[1, 3], lev, f"loader events (ev_pix/ev_pol): {len(pix)}")
    time_hist(ax[1, 4], tau * dt, pol, span, "loader event times (ev_tau x dt); dotted: hidden frames", mid_tau * dt)
    ax[0, 4].set_xlim(ax[1, 4].get_xlim()); ax[0, 4].set_ylim(top=max(ax[0, 4].get_ylim()[1], ax[1, 4].get_ylim()[1]))
    ax[1, 4].set_ylim(ax[0, 4].get_ylim())
    a = ax[1, 0]
    a.axis("off")
    g = float(np.asarray(item["gamma"])) if "gamma" in item else None
    lines = [f"sequence  {seqname}", f"frames    I0={info['i']}  I1={info['j']}  hidden={info['skip']}",
             f"context   {info['ctx']}", f"t0, t1    {info['t0']:.6f}, {info['t1']:.6f} s", f"dt        {dt * 1e3:.3f} ms",
             f"gamma     {g}", f"dataset id {int(np.asarray(item['ds']))}", "", "checks:"]
    mark = lambda ok: "-- " if ok is None else "OK " if ok else "XX "
    lines += [f"  {mark(ok)} {label}" + (f"  ({extra})" if extra else "") for label, ok, extra in checks]
    a.text(0, 1, "\n".join(lines), va="top", family="monospace", fontsize=9)
    ax[0, 0].set_ylabel("ORIGINAL DATASET", fontsize=12)
    ax[1, 1].set_ylabel("DATALOADER", fontsize=12)
    plt.tight_layout()
    plt.show()

    # context frames, hidden frames, voxel grid
    ctx, ctx_tau = np.asarray(item["ctx"]), np.asarray(item["ctx_tau"])
    mid = np.asarray(item["mid"])
    cols = max(len(ctx), len(mid) + 3)
    fig, ax = plt.subplots(2, cols, figsize=(2.6 * cols, 6))
    for k in range(cols):
        if k < len(ctx):
            show_img(ax[0, k], to_u8(ctx[k]), f"ctx tau {ctx_tau[k]:+.2f}")
        else:
            ax[0, k].axis("off")
        ax[1, k].axis("off")
    for k in range(len(mid)):
        ax[1, k].axis("on")
        show_img(ax[1, k], to_u8(mid[k]), f"hidden tau {mid_tau[k]:.2f}")
    B = vox.shape[0] // 2
    k = len(mid)
    for q, (lab, sl) in enumerate((("voxel ON (sum over bins)", slice(0, B)), ("voxel OFF (sum over bins)", slice(B, 2 * B)))):
        ax[1, k + q].axis("on")
        ax[1, k + q].imshow(np.log1p(vox[sl].sum(0)), cmap="magma")
        ax[1, k + q].set_title(lab, fontsize=9); ax[1, k + q].set_xticks([]); ax[1, k + q].set_yticks([])
    b = ax[1, k + 2]
    b.axis("on")
    b.bar(np.arange(B) - 0.2, vox[:B].sum((1, 2)), 0.4, color=(0.9, 0.2, 0.15), label="ON")
    b.bar(np.arange(B) + 0.2, vox[B:].sum((1, 2)), 0.4, color=(0.2, 0.45, 0.95), label="OFF")
    b.set_title("events per voxel time bin", fontsize=9); b.legend(fontsize=7)
    fig.suptitle(f"{name} | loader index {n} | context frames (student), hidden frames (photometric loss), voxel grid (teacher)",
                 fontsize=11)
    plt.tight_layout()
    plt.show()

    for s_ in notes:
        print("note:", s_)
    if raw is not None:
        print("original files:", *raw["files"], sep="\n  ")
    bad = [label for label, ok, _ in checks if ok is False]
    warn = [label for label, ok, _ in checks if ok is None]
    print(f"{name} sample {n}: " + ("all checks OK" if not bad else "CHECK FAILED: " + "; ".join(bad))
          + "".join(f" | NOTE: {w}" for w in warn))


def find_index(name, seq_name, frame):
    # loader index whose I0 is this frame of this sequence (or the nearest one)
    ds = DS[name]
    cand = [(abs(i - frame), n) for n, (k, i) in enumerate(ds.index) if os.path.basename(ds.seqs[k].root) == seq_name]
    if not cand:
        raise KeyError(f"{seq_name} not in {name}; sequences: {sorted({os.path.basename(s.root) for s in ds.seqs})}")
    return min(cand)[1]


def overview(name):
    ds = DS[name]
    print(ds.summary())
    start = 0
    for k, s in enumerate(ds.seqs):
        n = sum(1 for kk, _ in ds.index if kk == k)
        print(f"  {os.path.basename(s.root):32s} frames {len(s):6d}  {s.fps:6.1f} fps  skip {ds.skips[k]}  "
              f"loader index {start}..{start + n - 1}")
        start += n

In [ ]:
# ---- the four datasets, built exactly as train.py builds them --------------------------------------------------
cfg = yaml.safe_load(open(CONFIG))
D = dict(cfg["data"], crop=CROP or cfg["data"]["crop"])
sets = {s["name"]: s for s in data_sets(D, cfg["model"].get("gamma"))}
DS = {}
for name, p in DATA.items():
    root = os.path.join(p["conv"], SPLIT)
    if not os.path.isdir(root):
        print(f"{name}: {root} not found, skipped")
        continue
    s = sets.get(name) or data_sets(dict(D, root=root), cfg["model"].get("gamma"))[0]
    DS[name] = PairDataset(root, D["crop"], s["skip"], D["context"], D["bins"], s["min_events"] if TRAIN_MODE else 0,
                           TRAIN_MODE, gap=s["gap"], gamma=s["gamma"], ds=s["ds"], name=name, default_gamma=s["default_gamma"])
for name in DS:
    overview(name)

## HQ-EVFI

Original frames are the PNGs the converted frames link to; events are the per-interval `.npz` files between them.

In [ ]:
if "hqevfi" in DS:
    for n in INDEX["hqevfi"]:
        compare("hqevfi", n)

## BS-ERGB

Original frames are the PNGs the converted frames link to; events are `events/<k>.npz` (x, y in 1/32 px).

In [ ]:
if "bsergb" in DS:
    for n in INDEX["bsergb"]:
        compare("bsergb", n)

## CED

Original frames and events are read from the `.bag` (the first sample of a bag reads the whole bag once).

In [ ]:
if "ced" in DS:
    for n in INDEX["ced"]:
        compare("ced", n)

## EDS

Events are read from `events.h5` for the same time window. The stored RGB sees the scene through another lens, so the notebook warps it into the event camera with the dataset calibration (as the converter does) before cutting the crop; the full-frame panel shows that aligned view.

In [ ]:
if "eds" in DS:
    for n in INDEX["eds"]:
        compare("eds", n)

## Pick a sample by sequence and frame

`overview(name)` lists every sequence with its loader index range; `find_index(name, sequence, frame)` gives the loader
index whose I0 is that frame.

In [ ]:
# example: the sample starting nearest to frame 120 of the first sequence of the first dataset
name = next(iter(DS))
seq0 = os.path.basename(DS[name].seqs[0].root)
compare(name, find_index(name, seq0, 120))